In [ ]:
%sql
-- ============================================================
-- Restockify Workflow — §4.2 Deep Analysis, as Unity Catalog functions
-- ============================================================
-- Registers the Genie Agent's deep-analysis logic (consumption trend,
-- stockout forecast, urgency classification, quote line-item math, veto
-- inputs) as governed Unity Catalog SQL functions. The functions themselves
-- are registered in ab_training.agentic_restock (the schema we own), but
-- their bodies read Data Engineering's real gold_dev star schema:
--   - gold_dev.dim.dim_part / dim_warehouse / dim_plant / dim_supplier —
--     business-key lookups.
--   - gold_dev.supply_chain_analytics.fact_inventory_snapshot — current stock,
--     SAFETY_STOCK_QTY (reorder trigger), MAX_STOCK_LEVEL (restock target),
--     STOCKOUT_RISK. Daily snapshot grain, so callers always take the MOST
--     RECENT SNAPSHOT_DATE_KEY per part/warehouse (via MAX_BY), never a raw MAX().
--   - gold_dev.supply_chain_analytics.fact_inventory_transaction — ISSUE-type
--     rows are the consumption events (replaces the old mock consumption_history).
--   - gold_dev.supply_chain_analytics.fact_procurement — open purchase orders,
--     used by the restock-veto inputs (open_procurement_orders /
--     pending_procurement_qty) and by avg_lead_time_days.
--
-- Design note (v2 — tool granularity pass): there is deliberately no single
-- `needs_restock` boolean function. Earlier this repo had one, but a
-- yes/no veto is the one piece of reasoning that IS the Genie Agent's job
-- (architecture §4.2) — collapsing it into an opaque function meant Genie
-- could only echo TRUE/FALSE, never explain *why* (fully covered by an open
-- PO? partially? no plant link at all?). Instead, open_procurement_orders
-- and pending_procurement_qty expose the raw veto *inputs*; Genie composes
-- the actual veto decision by comparing pending_procurement_qty against
-- requested_restock_qty itself (see the Genie Space's text_instructions).
-- Same reasoning for restock_candidate_summary: kept, but narrowed in the
-- prompt to the single literal "why does X need restocking" phrasing —
-- everything else should be composed from the atomic functions below, not
-- forced through one canned template.
--
-- All functions keep an item_id/warehouse_id-shaped external signature (now
-- `part_id`/`warehouse_id`, since the gold_dev business keys are named
-- PART_ID/WAREHOUSE_ID) so Genie and the Supervisor Agent don't need to
-- reason about surrogate keys (PART_KEY/WAREHOUSE_KEY) at all.
--
-- Why UC functions instead of a notebook/job task: per Databricks Agent
-- Bricks docs, this is the correct primitive for "complex logic that
-- cannot be captured with a static or parameterized SQL query" — they can
-- be registered as trusted SQL functions on a Genie Agent, added directly
-- as tools on a Supervisor Agent, and queried by anyone with EXECUTE
-- permission, all without duplicating the logic in Python.
-- ============================================================

In [ ]:
%sql
-- ============================================================
-- Function 1: avg_daily_consumption
-- Trailing-window average daily consumption, anchored to today
-- (current_date()). Reads gold_dev.supply_chain_analytics.
-- fact_inventory_transaction directly (TRANSACTION_TYPE = 'ISSUE' rows
-- are consumption events) rather than fact_inventory_snapshot's
-- precomputed AVG_DAILY_CONSUMPTION column, so the `lookback_days`
-- parameter stays meaningful and auditable instead of trusting an
-- opaque, differently-windowed DE aggregate.
-- NOTE for this dev dataset: transaction rows are seeded for a fixed
-- 2026-08-11..2026-08-20 range, so as real time moves past that range
-- the window naturally covers fewer seed days — expected behavior for
-- a static dev dataset, not a bug. Kept as a single flat query (no
-- subqueries/window functions) because Databricks SQL functions reject
-- correlated subqueries once another function calls this one and its
-- body gets inlined.
-- ============================================================

CREATE OR REPLACE FUNCTION ab_training.agentic_restock.avg_daily_consumption(
  part_id STRING COMMENT 'Part business key, e.g. P1001 (gold_dev.dim.dim_part.PART_ID)',
  warehouse_id STRING COMMENT 'Warehouse business key, e.g. WH001 (gold_dev.dim.dim_warehouse.WAREHOUSE_ID)',
  lookback_days INT DEFAULT 14 COMMENT 'Trailing window size in days, ending today'
)
RETURNS DOUBLE
COMMENT 'Average daily consumption (architecture §4.2) over the trailing `lookback_days` ending today, computed from ISSUE-type rows in gold_dev.supply_chain_analytics.fact_inventory_transaction. Returns 0.0 if no consumption transactions exist in that window.'
RETURN
  SELECT COALESCE(SUM(fit.QUANTITY), 0.0) / avg_daily_consumption.lookback_days
  FROM gold_dev.supply_chain_analytics.fact_inventory_transaction fit
  JOIN gold_dev.dim.dim_part dp ON fit.PART_KEY = dp.PART_KEY
  JOIN gold_dev.dim.dim_warehouse dw ON fit.WAREHOUSE_KEY = dw.WAREHOUSE_KEY
  WHERE dp.PART_ID = avg_daily_consumption.part_id
    AND dw.WAREHOUSE_ID = avg_daily_consumption.warehouse_id
    AND fit.TRANSACTION_TYPE = 'ISSUE'
    AND to_date(CAST(fit.TRANSACTION_DATE_KEY AS STRING), 'yyyyMMdd') > date_sub(current_date(), avg_daily_consumption.lookback_days);

In [ ]:
%sql
-- ============================================================
-- Function 2: predicted_stockout_date
-- Forward-looking forecast from *today* (real time), using the most
-- recent snapshot's on-hand stock (fact_inventory_snapshot is a daily
-- snapshot fact, not a single current-state row, so MAX_BY picks the
-- latest SNAPSHOT_DATE_KEY per part/warehouse) and the trailing
-- avg_daily_consumption.
-- ============================================================

CREATE OR REPLACE FUNCTION ab_training.agentic_restock.predicted_stockout_date(
  part_id STRING COMMENT 'Part business key',
  warehouse_id STRING COMMENT 'Warehouse business key'
)
RETURNS DATE
COMMENT 'Earliest predicted stockout date (architecture §4.2), projected from today using the latest fact_inventory_snapshot QUANTITY_ON_HAND and the trailing 14-day avg_daily_consumption. NULL when consumption is ~0 (nothing to forecast).'
RETURN
  -- MAX(...) wrappers around nested function calls are required:
  -- Databricks SQL functions reject a table-scanning body whose SELECT
  -- list isn't provably single-row via aggregation once the function is
  -- called from inside another function's body — a PK-filtered WHERE
  -- alone isn't accepted as proof. MAX_BY(..., SNAPSHOT_DATE_KEY) is
  -- itself an aggregate, so it satisfies the same requirement while also
  -- picking the latest snapshot row.
  SELECT
    CASE
      WHEN MAX(ab_training.agentic_restock.avg_daily_consumption(predicted_stockout_date.part_id, predicted_stockout_date.warehouse_id, 14)) > 0
      THEN date_add(
        current_date(),
        CAST(CEIL(
          MAX_BY(fis.QUANTITY_ON_HAND, fis.SNAPSHOT_DATE_KEY)
          / MAX(ab_training.agentic_restock.avg_daily_consumption(predicted_stockout_date.part_id, predicted_stockout_date.warehouse_id, 14))
        ) AS INT)
      )
      ELSE NULL
    END
  FROM gold_dev.supply_chain_analytics.fact_inventory_snapshot fis
  JOIN gold_dev.dim.dim_part dp ON fis.PART_KEY = dp.PART_KEY
  JOIN gold_dev.dim.dim_warehouse dw ON fis.WAREHOUSE_KEY = dw.WAREHOUSE_KEY
  WHERE dp.PART_ID = predicted_stockout_date.part_id
    AND dw.WAREHOUSE_ID = predicted_stockout_date.warehouse_id;

In [ ]:
%sql
-- ============================================================
-- Function 3: classify_urgency
-- Pure classification, no table access. The gold_dev star schema has
-- no minimum_stock_qty-style absolute floor column, so the "always
-- CRITICAL regardless of forecast" override now uses Data Engineering's
-- own precomputed fact_inventory_snapshot.STOCKOUT_RISK = 'HIGH' signal
-- instead; otherwise urgency is banded by days_remaining until stockout,
-- same as before.
-- ============================================================

CREATE OR REPLACE FUNCTION ab_training.agentic_restock.classify_urgency(
  stockout_risk STRING COMMENT 'Latest fact_inventory_snapshot.STOCKOUT_RISK for this part/warehouse (LOW/MEDIUM/HIGH)',
  days_remaining DOUBLE COMMENT 'Days until predicted stockout, or NULL if no forecast (near-zero consumption)'
)
RETURNS STRING
COMMENT 'Urgency classification per architecture §4.2: CRITICAL (fact_inventory_snapshot.STOCKOUT_RISK = HIGH, or <=3 days to stockout), HIGH (<=7 days), MEDIUM (<=14 days), LOW (>14 days or no forecastable consumption).'
RETURN
  CASE
    WHEN classify_urgency.stockout_risk = 'HIGH' THEN 'CRITICAL'
    WHEN classify_urgency.days_remaining IS NULL THEN 'LOW'
    WHEN classify_urgency.days_remaining <= 3 THEN 'CRITICAL'
    WHEN classify_urgency.days_remaining <= 7 THEN 'HIGH'
    WHEN classify_urgency.days_remaining <= 14 THEN 'MEDIUM'
    ELSE 'LOW'
  END;

In [ ]:
%sql
-- ============================================================
-- Function 4: requested_restock_qty
-- Quote line-item math: how many units to order to reach the restock
-- target. Floored at 0 (never a negative order). MAX_STOCK_LEVEL takes
-- over the role the old mock threshold_config_table.target_stock_qty
-- played; both come from the latest fact_inventory_snapshot row.
-- ============================================================

CREATE OR REPLACE FUNCTION ab_training.agentic_restock.requested_restock_qty(
  part_id STRING COMMENT 'Part business key',
  warehouse_id STRING COMMENT 'Warehouse business key'
)
RETURNS INT
COMMENT 'Suggested restock quantity: MAX_STOCK_LEVEL - QUANTITY_ON_HAND (from the latest fact_inventory_snapshot row), floored at 0. NULL if the part/warehouse has no snapshot rows.'
RETURN
  -- No outer MAX(...) wrapper here: GREATEST(...)'s only arguments are the
  -- two MAX_BY(...) aggregates below plus a literal 0, so the expression is
  -- already a single top-level aggregate result (no GROUP BY needed).
  -- Wrapping it in another MAX(...) is what Spark's NESTED_AGGREGATE_FUNCTION
  -- check rejects -- aggregate-of-aggregate in the same query, not just a
  -- table scan being proven single-row.
  SELECT GREATEST(
    MAX_BY(fis.MAX_STOCK_LEVEL, fis.SNAPSHOT_DATE_KEY) - MAX_BY(fis.QUANTITY_ON_HAND, fis.SNAPSHOT_DATE_KEY),
    0
  )
  FROM gold_dev.supply_chain_analytics.fact_inventory_snapshot fis
  JOIN gold_dev.dim.dim_part dp ON fis.PART_KEY = dp.PART_KEY
  JOIN gold_dev.dim.dim_warehouse dw ON fis.WAREHOUSE_KEY = dw.WAREHOUSE_KEY
  WHERE dp.PART_ID = requested_restock_qty.part_id
    AND dw.WAREHOUSE_ID = requested_restock_qty.warehouse_id;

In [ ]:
%sql
-- ============================================================
-- Function 5a: pending_procurement_qty (veto input, scalar)
-- Raw veto *input*, not the veto decision itself -- see the design note in
-- the header cell above. Total PENDING_QTY across open (ISSUED/PARTIAL)
-- purchase orders for this part at the warehouse's linked plant. Genie
-- compares this against requested_restock_qty itself to decide whether a
-- Lakeflow-flagged candidate is a false positive (already covered by an
-- in-flight PO) or genuinely needs restocking, and can explain the gap
-- (e.g. "PO covers 800 of the 1690 needed") instead of a bare yes/no.
-- Returns 0.0 both when there's no open PO and when the warehouse has no
-- linked plant at all (e.g. REGIONAL SPARES warehouses per
-- dim_warehouse.LINKED_PLANT_ID) -- use open_procurement_orders below, or a
-- direct dim_warehouse lookup, to tell those two cases apart if needed.
-- ============================================================

-- Clean up the old opaque boolean veto function this notebook used to
-- define (see the v2 design note in the header cell) -- it has no
-- replacement of the same name, so CREATE OR REPLACE below wouldn't drop it.
DROP FUNCTION IF EXISTS ab_training.agentic_restock.needs_restock;

CREATE OR REPLACE FUNCTION ab_training.agentic_restock.pending_procurement_qty(
  part_id STRING COMMENT 'Part business key',
  warehouse_id STRING COMMENT 'Warehouse business key'
)
RETURNS DOUBLE
COMMENT 'Veto input (architecture §4.2), not the veto decision: total PENDING_QTY across open (ISSUED/PARTIAL) purchase orders for this part at the warehouse''s linked plant, from gold_dev.supply_chain_analytics.fact_procurement. 0.0 if there is no open PO, or if the warehouse has no linked plant at all. Compare against requested_restock_qty yourself -- if pending_procurement_qty already covers or exceeds it, restocking may already be in flight (a false positive); otherwise it is genuinely needed.'
RETURN
  SELECT COALESCE(SUM(fp.PENDING_QTY), 0.0)
  FROM gold_dev.dim.dim_warehouse dw
  LEFT JOIN gold_dev.dim.dim_plant dpl
    ON dpl.PLANT_ID = dw.LINKED_PLANT_ID
  LEFT JOIN gold_dev.dim.dim_part dp
    ON dp.PART_ID = pending_procurement_qty.part_id
  LEFT JOIN gold_dev.supply_chain_analytics.fact_procurement fp
    ON fp.PLANT_KEY = dpl.PLANT_KEY
    AND fp.PART_KEY = dp.PART_KEY
    AND fp.STATUS IN ('ISSUED', 'PARTIAL')
  WHERE dw.WAREHOUSE_ID = pending_procurement_qty.warehouse_id;

In [ ]:
%sql
-- ============================================================
-- Function 5b: open_procurement_orders (veto input, table-valued)
-- The row-level companion to pending_procurement_qty: the actual open POs
-- (largest PENDING_QTY first) instead of just their sum, so Genie can name
-- a specific PO / supplier / expected date in its explanation ("covered by
-- PO-... from ... expected ...") rather than a bare number. Same open
-- (ISSUED/PARTIAL), linked-plant join as pending_procurement_qty -- an
-- empty result set means either no open PO, or no linked plant at all.
-- ============================================================

CREATE OR REPLACE FUNCTION ab_training.agentic_restock.open_procurement_orders(
  part_id STRING COMMENT 'Part business key',
  warehouse_id STRING COMMENT 'Warehouse business key'
)
RETURNS TABLE (
  purchase_order_id STRING COMMENT 'PO business key, e.g. PO-2026-00143',
  status STRING COMMENT 'ISSUED or PARTIAL',
  pending_qty INT COMMENT 'Quantity not yet received on this PO',
  expected_date DATE COMMENT 'Expected delivery date',
  supplier_name STRING COMMENT 'Supplier legal name, if known'
)
COMMENT 'Veto input (architecture §4.2), not the veto decision: open (ISSUED/PARTIAL) purchase orders for this part at the warehouse''s linked plant, from gold_dev.supply_chain_analytics.fact_procurement, largest pending_qty first. Empty result means either no open PO was found or the warehouse has no linked plant at all (dim_warehouse.LINKED_PLANT_ID is NULL). Sum pending_qty and compare against requested_restock_qty yourself to decide whether restocking is already covered.'
RETURN
  SELECT
    fp.PURCHASE_ORDER_ID AS purchase_order_id,
    fp.STATUS AS status,
    fp.PENDING_QTY AS pending_qty,
    to_date(CAST(fp.EXPECTED_DATE_KEY AS STRING), 'yyyyMMdd') AS expected_date,
    ds.SUPPLIER_NAME AS supplier_name
  FROM gold_dev.dim.dim_warehouse dw
  JOIN gold_dev.dim.dim_plant dpl
    ON dpl.PLANT_ID = dw.LINKED_PLANT_ID
  JOIN gold_dev.dim.dim_part dp
    ON dp.PART_ID = open_procurement_orders.part_id
  JOIN gold_dev.supply_chain_analytics.fact_procurement fp
    ON fp.PLANT_KEY = dpl.PLANT_KEY
    AND fp.PART_KEY = dp.PART_KEY
    AND fp.STATUS IN ('ISSUED', 'PARTIAL')
  LEFT JOIN gold_dev.dim.dim_supplier ds
    ON ds.SUPPLIER_KEY = fp.SUPPLIER_KEY
    AND ds.IS_CURRENT = true
  WHERE dw.WAREHOUSE_ID = open_procurement_orders.warehouse_id
  ORDER BY fp.PENDING_QTY DESC;

-- Standalone verification: open POs (if any) for a sample candidate. Uses a
-- literal part_id/warehouse_id (like the other functions' verification
-- cells) rather than a LATERAL join against a driving query -- Databricks
-- SQL rejects the LATERAL correlation once the function's own body reuses
-- the same table alias (dp/dw) as the outer query.
SELECT * FROM ab_training.agentic_restock.open_procurement_orders('P1003', 'WH003');

In [ ]:
%sql
-- ============================================================
-- Function 6: restock_candidate_summary
-- Deterministic natural-language one-liner combining functions
-- 1-4. Deliberately narrow scope (see the v2 design note in the header
-- cell): this is a canned template for the single literal phrasing "why
-- does X need restocking" / the Teams Adaptive Card + quote_metadata.
-- summary_report text source. It is NOT Genie's general-purpose
-- explanation tool -- comparisons, "what if", and multi-candidate
-- questions should be composed from the atomic functions instead of
-- forced through this one template (see the Genie Space's
-- text_instructions).
-- NOTE: gold_dev has no unit_of_measure column on dim_part or
-- fact_inventory_snapshot (unlike the old mock inventory_stock_level),
-- so quantities are reported as plain "units" here.
-- ============================================================

CREATE OR REPLACE FUNCTION ab_training.agentic_restock.restock_candidate_summary(
  part_id STRING COMMENT 'Part business key',
  warehouse_id STRING COMMENT 'Warehouse business key'
)
RETURNS STRING
COMMENT 'Deterministic natural-language summary of one restock candidate: stock on hand, avg daily consumption, predicted stockout date, urgency, and suggested reorder quantity (architecture §4.2).'
RETURN
  -- A single-row CTE does all the aggregation up front (MAX_BY for the latest
  -- snapshot fields, MAX(...) for the otherwise-plain dp.PART_NAME/
  -- dw.WAREHOUSE_CODE columns so they're valid alongside the aggregates with
  -- no GROUP BY). The final SELECT then reads plain CTE columns -- no
  -- aggregate wrapped around another aggregate, which is what
  -- NESTED_AGGREGATE_FUNCTION rejects.
  WITH latest AS (
    SELECT
      MAX(dp.PART_NAME) AS part_name,
      MAX(dw.WAREHOUSE_CODE) AS warehouse_code,
      MAX_BY(fis.QUANTITY_ON_HAND, fis.SNAPSHOT_DATE_KEY) AS quantity_on_hand,
      MAX_BY(fis.SAFETY_STOCK_QTY, fis.SNAPSHOT_DATE_KEY) AS safety_stock_qty,
      MAX_BY(fis.STOCKOUT_RISK, fis.SNAPSHOT_DATE_KEY) AS stockout_risk
    FROM gold_dev.supply_chain_analytics.fact_inventory_snapshot fis
    JOIN gold_dev.dim.dim_part dp ON fis.PART_KEY = dp.PART_KEY
    JOIN gold_dev.dim.dim_warehouse dw ON fis.WAREHOUSE_KEY = dw.WAREHOUSE_KEY
    WHERE dp.PART_ID = restock_candidate_summary.part_id
      AND dw.WAREHOUSE_ID = restock_candidate_summary.warehouse_id
  )
  SELECT CONCAT(
    latest.part_name, ' at ', latest.warehouse_code, ': ',
    CAST(latest.quantity_on_hand AS STRING), ' units on hand (safety stock ',
    CAST(latest.safety_stock_qty AS STRING), '). Avg consumption ',
    CAST(ROUND(ab_training.agentic_restock.avg_daily_consumption(restock_candidate_summary.part_id, restock_candidate_summary.warehouse_id, 14), 1) AS STRING),
    '/day. ',
    CASE
      WHEN ab_training.agentic_restock.predicted_stockout_date(restock_candidate_summary.part_id, restock_candidate_summary.warehouse_id) IS NOT NULL
      THEN CONCAT('Predicted stockout ', CAST(ab_training.agentic_restock.predicted_stockout_date(restock_candidate_summary.part_id, restock_candidate_summary.warehouse_id) AS STRING), '. ')
      ELSE 'No forecastable stockout (near-zero consumption). '
    END,
    'Urgency: ', ab_training.agentic_restock.classify_urgency(
      latest.stockout_risk,
      datediff(ab_training.agentic_restock.predicted_stockout_date(restock_candidate_summary.part_id, restock_candidate_summary.warehouse_id), current_date())
    ), '. ',
    'Suggested reorder: ', CAST(ab_training.agentic_restock.requested_restock_qty(restock_candidate_summary.part_id, restock_candidate_summary.warehouse_id) AS STRING), ' units.'
  )
  FROM latest;

In [ ]:
%sql
-- ============================================================
-- Verification: run the core functions against every candidate the
-- §4.1 coarse check would flag (QUANTITY_ON_HAND <= SAFETY_STOCK_QTY on
-- the latest fact_inventory_snapshot row per part/warehouse). This
-- mirrors src/agentic_restock/jobs/lakeflow_trigger.py's
-- build_coarse_check_query() -- eyeball these against the real gold_dev
-- data seeded by Data Engineering. requested_restock_qty vs.
-- pending_procurement_qty is the veto comparison Genie is expected to
-- make itself (architecture §4.2) -- there is no single needs_restock
-- boolean function anymore, see the v2 design note in the header cell.
-- ============================================================

WITH latest_snapshot AS (
  SELECT
    *,
    ROW_NUMBER() OVER (PARTITION BY PART_KEY, WAREHOUSE_KEY ORDER BY SNAPSHOT_DATE_KEY DESC) AS rn
  FROM gold_dev.supply_chain_analytics.fact_inventory_snapshot
)
SELECT
  dp.PART_ID AS part_id,
  dp.PART_NAME AS part_name,
  dw.WAREHOUSE_ID AS warehouse_id,
  ls.QUANTITY_ON_HAND AS current_stock_qty,
  ls.SAFETY_STOCK_QTY AS reorder_point_qty,
  ls.STOCKOUT_RISK AS stockout_risk,
  ROUND(ab_training.agentic_restock.avg_daily_consumption(dp.PART_ID, dw.WAREHOUSE_ID, 14), 2) AS avg_daily_consumption,
  ab_training.agentic_restock.predicted_stockout_date(dp.PART_ID, dw.WAREHOUSE_ID) AS predicted_stockout_date,
  ab_training.agentic_restock.classify_urgency(
    ls.STOCKOUT_RISK,
    datediff(ab_training.agentic_restock.predicted_stockout_date(dp.PART_ID, dw.WAREHOUSE_ID), current_date())
  ) AS urgency_level,
  ab_training.agentic_restock.requested_restock_qty(dp.PART_ID, dw.WAREHOUSE_ID) AS requested_restock_qty,
  ab_training.agentic_restock.pending_procurement_qty(dp.PART_ID, dw.WAREHOUSE_ID) AS pending_procurement_qty,
  ab_training.agentic_restock.restock_candidate_summary(dp.PART_ID, dw.WAREHOUSE_ID) AS summary
FROM latest_snapshot ls
JOIN gold_dev.dim.dim_part dp ON ls.PART_KEY = dp.PART_KEY AND dp.IS_CURRENT = true
JOIN gold_dev.dim.dim_warehouse dw ON ls.WAREHOUSE_KEY = dw.WAREHOUSE_KEY
WHERE ls.rn = 1
  AND dp.LIFECYCLE_STATUS = 'ACTIVE'
  AND dw.OPERATIONAL_STATUS = 'ACTIVE'
  AND ls.QUANTITY_ON_HAND <= ls.SAFETY_STOCK_QTY
ORDER BY urgency_level, part_id;

In [ ]:
%sql
-- ============================================================
-- Function 7 (new): avg_lead_time_days
-- The old mock threshold_config_table.lead_time_days config field has no
-- equivalent anywhere in the gold_dev star schema -- dim_part, dim_supplier,
-- and fact_procurement were all checked and none carry a fixed lead-time
-- config. This function derives an *empirical* estimate instead, from
-- historical purchase orders: EXPECTED_DATE_KEY - ORDER_DATE_KEY, averaged
-- across all of a part's POs (any supplier/plant). Not a contracted SLA --
-- purely informational, surfaced in Teams/Genie text, not used by
-- classify_urgency or the veto. Returns NULL if the part has no
-- procurement history.
-- ============================================================

CREATE OR REPLACE FUNCTION ab_training.agentic_restock.avg_lead_time_days(
  part_id STRING COMMENT 'Part business key, e.g. P1001'
)
RETURNS DOUBLE
COMMENT 'Empirical average supplier lead time in days for a part, derived from gold_dev.supply_chain_analytics.fact_procurement (EXPECTED_DATE_KEY - ORDER_DATE_KEY, averaged across all historical POs for the part). Not a contracted SLA -- the gold_dev star schema has no fixed lead_time_days config field. NULL if the part has no procurement history.'
RETURN
  SELECT AVG(
    datediff(
      to_date(CAST(fp.EXPECTED_DATE_KEY AS STRING), 'yyyyMMdd'),
      to_date(CAST(fp.ORDER_DATE_KEY AS STRING), 'yyyyMMdd')
    )
  )
  FROM gold_dev.supply_chain_analytics.fact_procurement fp
  JOIN gold_dev.dim.dim_part dp ON fp.PART_KEY = dp.PART_KEY
  WHERE dp.PART_ID = avg_lead_time_days.part_id;

-- Standalone verification: average lead time for the first 5 parts with
-- procurement history.
SELECT DISTINCT
  dp.PART_ID,
  dp.PART_NAME,
  ROUND(ab_training.agentic_restock.avg_lead_time_days(dp.PART_ID), 1) AS avg_lead_time_days
FROM gold_dev.supply_chain_analytics.fact_procurement fp
JOIN gold_dev.dim.dim_part dp ON fp.PART_KEY = dp.PART_KEY
ORDER BY dp.PART_ID
LIMIT 5;

In [ ]:
%sql
-- ============================================================
-- Function 8 (new): latest_snapshot
-- fact_inventory_snapshot is a daily snapshot fact (one row per part x
-- warehouse x day), not a single current-state row -- every other
-- function in this notebook dedups via MAX_BY(..., SNAPSHOT_DATE_KEY).
-- Genie has direct read access to fact_inventory_snapshot too (it's a
-- Genie Space data source), so for a plain "what's the current stock of
-- X" question it could generate ad-hoc SQL that forgets this dedup and
-- silently double-counts or picks a stale day. This function makes the
-- correct, single-row answer a trusted asset instead, so that failure
-- mode can't happen for the most common question shape.
-- ============================================================

CREATE OR REPLACE FUNCTION ab_training.agentic_restock.latest_snapshot(
  part_id STRING COMMENT 'Part business key',
  warehouse_id STRING COMMENT 'Warehouse business key'
)
RETURNS TABLE (
  snapshot_date DATE COMMENT 'Date of this snapshot row (the most recent available)',
  quantity_on_hand INT COMMENT 'Units currently on hand',
  safety_stock_qty INT COMMENT 'Reorder trigger point',
  max_stock_level INT COMMENT 'Restock target',
  stockout_risk STRING COMMENT 'LOW / MEDIUM / HIGH, per Data Engineering'
)
COMMENT 'The single most recent gold_dev.supply_chain_analytics.fact_inventory_snapshot row for a part/warehouse, already deduped on MAX(SNAPSHOT_DATE_KEY). Use this instead of querying fact_inventory_snapshot directly for "what is the current stock" style questions -- a plain SELECT without this dedup can pick a stale day since this fact table is a daily snapshot, not a single current-state row. Empty result means the part/warehouse has no snapshot rows at all.'
RETURN
  SELECT
    to_date(CAST(fis.SNAPSHOT_DATE_KEY AS STRING), 'yyyyMMdd') AS snapshot_date,
    fis.QUANTITY_ON_HAND AS quantity_on_hand,
    fis.SAFETY_STOCK_QTY AS safety_stock_qty,
    fis.MAX_STOCK_LEVEL AS max_stock_level,
    fis.STOCKOUT_RISK AS stockout_risk
  FROM gold_dev.supply_chain_analytics.fact_inventory_snapshot fis
  JOIN gold_dev.dim.dim_part dp ON fis.PART_KEY = dp.PART_KEY
  JOIN gold_dev.dim.dim_warehouse dw ON fis.WAREHOUSE_KEY = dw.WAREHOUSE_KEY
  WHERE dp.PART_ID = latest_snapshot.part_id
    AND dw.WAREHOUSE_ID = latest_snapshot.warehouse_id
  QUALIFY ROW_NUMBER() OVER (ORDER BY fis.SNAPSHOT_DATE_KEY DESC) = 1;

-- Standalone verification: latest snapshot for a sample candidate. Uses a
-- literal part_id/warehouse_id rather than a LATERAL join (see the note in
-- the open_procurement_orders verification cell above for why).
SELECT * FROM ab_training.agentic_restock.latest_snapshot('P1003', 'WH003');